Data cleaning and ingredient normalisation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import json
import os

# Define in and out paths
DATA_PATH = "/content/drive/MyDrive/epi_r.csv"
OUTPUT_PATH = "/content/drive/MyDrive/known_ingredients.json"

# Checks if the outpath already exists. If so, skips, if not, it creates the list.
if os.path.exists(OUTPUT_PATH):
    # Prints output for checking (this would not be shown in the user's version).
    #print(f"{OUTPUT_PATH} already exists — skipping ingredient extraction.")

    with open(OUTPUT_PATH, "r") as f:
        ingredient_data = json.load(f)

    # Prints output for checking (this would not be shown in the user's version).
    normalised_ingredients = set(ingredient_data["normalised_ingredients"])
    #print(f"Loaded existing ingredient vocabulary ({len(normalised_ingredients)} ingredients)")

else:

    recipes = pd.read_csv(DATA_PATH)

    # Defines numeric columns (macros).
    numeric_cols = ['rating', 'calories', 'protein', 'fat', 'sodium']

    # Convert numeric columns.
    for col in numeric_cols:
        recipes[col] = pd.to_numeric(recipes[col], errors='coerce').fillna(0)

    # Remove low-calorie recipes to encourage safe eating.
    recipes = recipes[recipes['calories'] >= 300]

    # Fill NaNs with blank strings.
    recipes = recipes.fillna("")

    # Define a list of tags in the dataset which do not refer to ingredients, nor dish type (paella, curry, etc.).
    non_ingredients = [
        "#cakeweek", "#wasteless", "22-minute meals", "3-ingredient recipes", "30 days of groceries",
        "advance prep required", "alabama", "alaska", "alcoholic", "anniversary", "anthony bourdain",
        "aperitif", "aspen", "atlanta", "australia", "back to school", "backyard bbq", "bake",
        "bastille day", "beverly hills", "birthday", "blender", "boil", "bon appétit", "boston",
        "brunch", "buffalo", "buffet", "bulgaria", "california", "cambridge", "camping", "canada",
        "candy thermometer", "chicago", "christmas", "christmas eve", "cinco de mayo", "dallas",
        "deep-fry", "denver", "dessert", "digestif", "dinner", "diwali", "dominican republic",
        "dorie greenspan", "double boiler", "drink", "drinks", "easter", "eau de vie", "edible gift",
        "egypt", "emeril lagasse", "engagement party", "england", "entertaining", "epi + ushg",
        "epi loves the microwave", "fall", "family reunion", "fat free", "father's day",
        "flaming hot summer", "florida", "food processor", "fourth of july", "france", "freeze/chill",
        "freezer food", "friendsgiving", "frozen dessert", "game", "georgia", "germany", "gin",
        "gourmet", "graduation", "grains", "halloween", "hanukkah", "harpercollins", "hawaii",
        "healthy", "high fiber", "hollywood", "houston", "ice cream machine", "iced coffee", "iced tea",
        "idaho", "illinois", "indiana", "iowa", "ireland", "israel", "italy", "jamaica", "japan",
        "juicer", "kansas", "kansas city", "kentucky", "kentucky derby", "kid-friendly", "kidney friendly",
        "kitchen olympics", "kosher", "kosher for passover", "labor day", "las vegas", "lancaster",
        "leaf green", "louisiana", "louisville", "low cal", "low carb", "low cholesterol", "low fat",
        "low sodium", "low sugar", "low/no sugar", "lunar new year", "lunch", "maine", "mandoline",
        "maryland", "massachusetts", "miami", "michigan", "microwave", "minneapolis", "minnesota",
        "mississippi", "missouri", "mixer", "mother's day", "new hampshire", "new jersey", "new mexico",
        "new orleans", "new year's day", "new year's eve", "new york", "no meat, no problem", "no sugar added",
        "no-cook", "north carolina", "ohio", "oklahoma", "okterberfest", "one-pot meal", "pacific palisades",
        "paleo", "pan-fry", "pancake", "parade", "paris", "party", "pasadena", "passover", "pasta maker",
        "picnic", "pizza", "poblano", "poker/game night", "puerto rico", "quick & easy", "quick and healthy",
        "ramadan", "ramekin", "raw", "roast", "rosh hashanah/yom kippur", "rub", "salad", "salad dressing",
        "san francisco", "sandwich theory", "sangria", "santa monica", "saute", "seattle", "self", "shaker",
        "shavuot", "shower", "side", "simmer", "skewer", "slow cooker", "smoker", "smoothie", "st. louis",
        "st. patrick's day", "steam", "stir-fry", "stock", "stuffing/dressing", "sugar conscious", "sukkot",
        "summer", "super bowl", "suzanne goin", "tested & improved", "thanksgiving", "texas", "tuesday",
        "valentine's day", "vegan", "vegetarian", "virginia", "washington", "washington, d.c.", "wedding",
        "weelicious", "west virginia", "westwood", "wok", "cookbooks", "leftovers", "snack", "snack week", "winter",
        "dairy free", "peanut free", "soup/stew", "braise", "sauté", "cocktail party", "pescatarian", "soy free",
        "tree nut free", "wheat/gluten-free", "spring", "summer", "fall", "autumn", "casserole/gratin", "potluck",
        "sandwich", "grill/barbecque", "breakfast", "house & garden"
    ]

    # Extracts all columns referring to ingredients.
    feature_cols = [
        col for col in recipes.columns
        if col not in numeric_cols + ['title'] + non_ingredients
    ]

    raw_ingredients = set([col.lower().strip() for col in feature_cols])

    # Prints output for checking (this would not be shown in the user's version).
    # print(f"Found {len(raw_ingredients)} raw ingredient features")

    # Standardise ingredient name; removing upper cases, plurals and compounded ingredients.
    def normalise_ingredient(ing):
        ing = ing.lower().strip()

        ing = ing.replace("potatoes", "potato")
        ing = ing.replace("tomatoes", "tomato")

        if "/" in ing:
            return [p.strip() for p in ing.split("/")]

        return [ing]

    normalised_ingredients = set()

    for ing in raw_ingredients:
        normalised_ingredients.update(normalise_ingredient(ing))

    normalised_ingredients = sorted(normalised_ingredients)

    # Prints output for checking (this would not be shown in the user's version).
    # print(f"Normalised to {len(normalised_ingredients)} ingredients")

    # Save resultant list.
    ingredient_data = {
        "raw_ingredients": sorted(raw_ingredients),
        "normalised_ingredients": normalised_ingredients
    }

    with open(OUTPUT_PATH, "w") as f:
        json.dump(ingredient_data, f, indent=2)

    # Prints output for checking (this would not be shown in the user's version).
    # print(f"Saved ingredient vocabulary to: {OUTPUT_PATH}")

# Clean Clean epi_r.csv to give a series of single text strings which contain:
# Title | nutrition information (macros) | ingredient/dish type list

recipes = pd.read_csv("/content/drive/MyDrive/epi_r.csv")

# Define numeric columns to keep as numbers.
numeric_cols = ['rating', 'calories', 'protein', 'fat', 'sodium']

# Convert numeric columns to numbers, filling missing with 0.
for col in numeric_cols:
    recipes[col] = pd.to_numeric(recipes[col], errors='coerce').fillna(0)

# Remove recipes with calories under 300 to encourage safe eating habits.
recipes = recipes[recipes['calories'] >= 200]

# Removes rows without titles.
recipes = recipes[recipes['title'].notna()]
recipes = recipes[recipes['title'].str.strip() != ""]

# Defines list of non-ingredient/dish type tags to remove to reduce noise.
non_ingredients = [
    "#cakeweek", "#wasteless", "22-minute meals", "3-ingredient recipes", "30 days of groceries",
    "advance prep required", "alabama", "alaska", "alcoholic", "anniversary", "anthony bourdain",
    "aperitif", "aspen", "atlanta", "australia", "back to school", "backyard bbq", "bake",
    "bastille day", "beverly hills", "birthday", "blender", "boil", "bon appétit", "boston",
    "brunch", "buffalo", "buffet", "bulgaria", "california", "cambridge", "camping", "canada",
    "candy thermometer", "chicago", "christmas", "christmas eve", "cinco de mayo", "dallas",
    "deep-fry", "denver", "dessert", "digestif", "dinner", "diwali", "dominican republic",
    "dorie greenspan", "double boiler", "drink", "drinks", "easter", "eau de vie", "edible gift",
    "egypt", "emeril lagasse", "engagement party", "england", "entertaining",
    "epi + ushg", "epi loves the microwave", "fall", "family reunion", "fat free",
    "father's day", "flaming hot summer", "florida", "food processor", "fourth of july",
    "france", "freeze/chill", "freezer food", "friendsgiving", "frozen dessert", "game",
    "georgia", "germany", "gin", "gourmet", "graduation", "grains", "halloween",
    "hanukkah", "harpercollins", "hawaii", "healthy", "high fiber", "hollywood",
    "houston", "ice cream machine", "iced coffee", "iced tea", "idaho", "illinois",
    "indiana", "iowa", "ireland", "israel", "italy", "jamaica", "japan", "juicer",
    "kansas", "kansas city", "kentucky", "kentucky derby", "kid-friendly", "kidney friendly",
    "kitchen olympics", "kosher", "kosher for passover", "labor day", "las vegas",
    "lancaster", "leaf green", "louisiana", "louisville", "low cal", "low carb",
    "low cholesterol", "low fat", "low sodium", "low sugar", "low/no sugar",
    "lunar new year", "lunch", "maine", "mandoline", "maryland", "massachusetts",
    "miami", "michigan", "microwave", "minneapolis", "minnesota", "mississippi",
    "missouri", "mixer", "mother's day", "new hampshire", "new jersey", "new mexico",
    "new orleans", "new year's day", "new year's eve", "new york", "no meat, no problem",
    "no sugar added", "no-cook", "north carolina", "ohio", "oklahoma", "okterberfest",
    "one-pot meal", "pacific palisades", "paleo", "pan-fry", "parade", "paris",
    "party", "pasadena", "passover", "pasta maker", "picnic", "poblano",
    "poker/game night", "puerto rico", "quick & easy", "quick and healthy",
    "ramadan", "ramekin", "raw", "rosh hashanah/yom kippur", "rub", "salad",
    "salad dressing", "san francisco", "sandwich theory", "sangria", "santa monica",
    "saute", "seattle", "self", "shaker", "shavuot", "shower", "side", "simmer",
    "slow cooker", "smoker", "smoothie", "st. louis", "st. patrick's day", "steam",
    "stir-fry", "stock", "stuffing/dressing", "sugar conscious", "sukkot", "summer",
    "super bowl", "suzanne goin", "tested & improved", "thanksgiving", "texas",
    "tuesday", "valentine's day", "vegan", "vegetarian", "virginia", "washington",
    "washington, d.c.", "wedding", "weelicious", "west virginia", "westwood",
    "wok", "cookbooks", "leftovers", "snack", "snack week", "winter",
    "dairy free", "peanut free", "soup/stew", "braise", "cocktail party",
    "pescatarian", "soy free", "tree nut free", "wheat/gluten-free",
    "spring", "autumn", "casserole/gratin", "potluck", "sandwich",
    "grill/barbecque", "breakfast", "house & garden", "coffee grinder", "roast"
]

# Converts each row into a string of: "Title | nutrition information (macros) | ingredient/dish type list"
def row_to_text_with_ingredients(row):

    # Macro definition.
    numeric_str = " ".join(f"{col}: {row[col]}" for col in numeric_cols)

    # Ingredient/dish type extraction and joining.
    features = [
        col for col, val in row.items()
        if val == 1.0 and col not in numeric_cols + ['title'] + non_ingredients
    ]

    features_str = ", ".join(features)

    return f"{row['title']} | {numeric_str} | Features: {features_str}"

# Joins each string to a list.
texts = recipes.apply(row_to_text_with_ingredients, axis=1).tolist()

# Inspects first 5 rows.
# Prints output for checking (this would not be shown in the user's version).
# for i, text in enumerate(recipes.head(2).apply(row_to_text_with_ingredients, axis=1), 1):
    # print(f"Row {i}:\n{text}\n")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Embeds the recipe strings from the cleaned epi_r.csv.


In [ ]:
import os
import torch
import json
from sentence_transformers import SentenceTransformer, util, CrossEncoder

# Lightweight model.
model_name = "multi-qa-MiniLM-L6-cos-v1"
model = SentenceTransformer(model_name)

# The cached embedding path.
paragraph_path = "/content/drive/MyDrive/Embeddings/paragraph_embeddings.pt"

# Checks if embeddings have been cached and skips if so. If not, recipe strings from the cleaned epi_r.csv are embedded.
if os.path.exists(paragraph_path):
    paragraph_embeddings = torch.load(paragraph_path)
else:
    paragraph_embeddings = model.encode(texts, convert_to_tensor=True)
    torch.save(paragraph_embeddings, paragraph_path)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hybrid recipe search engine using:


*   Cosine similarity search
*   Multi-criterion scoring based on nutrition goals
*   Cross-encoder scoring
*   Ingredient overlap scoring


In [ ]:
# Loads the pre-defined known ingredients list.
KNOWN_INGREDIENTS_PATH = "/content/drive/MyDrive/known_ingredients.json"
with open(KNOWN_INGREDIENTS_PATH, "r") as f:
    known_data = json.load(f)

# change this z to s
normalised_ingredients = set(known_data["normalised_ingredients"])

# Creates a list of recipes in structured strings.
texts = recipes.apply(row_to_text_with_ingredients, axis=1).tolist()
recipes = recipes.reset_index(drop=True)

# User is asked to input a query of what they want.
query = input("Please tell me any ingredients you would like to use, the type of dish you would like and any nutritional goals you want to adhere to:\n").lower()

# Asks about dietary restrictions.
diet_input = input(
    "Please state all dietary preferences you have (e.g. peanut, gluten-free, vegan). If none, leave blank or say no:\n"
).strip().lower()

# Defines a manual list of dietary restrictions to enable strict, constraint-based filtering.
# Note: this has not been completely created yet but it works as a proof of concept.
expanded_terms = []
if diet_input not in ["no", "none", ""]:
    diet_terms = [t.strip() for t in diet_input.split(",") if t.strip()]
    allergy_map = {
        "nuts": ["nut", "nuts", "almond", "cashew", "walnut", "pecan", "hazelnut", "macadamia", "pistachio", "brazil nut"],
        "peanut": ["peanut", "peanuts", "groundnut"],
        "gluten": ["gluten", "wheat", "barley", "rye", "spelt", "kamut", "malt", "triticale", "bread", "sandwhich", "flour", "gluten-free"],
        "dairy": ["milk", "cheese", "butter", "cream", "yogurt", "ghee", "casein", "whey"],
        "soy": ["soy", "soya", "edamame", "tofu", "soy sauce", "tempeh", "soy"],
        "egg": ["egg", "eggs", "egg white", "egg yolk", "mayonnaise"],
        "shellfish": ["shrimp", "prawn", "crab", "lobster", "crayfish", "scallop", "clams", "mollusk"],
        "fish": ["salmon", "tuna", "cod", "trout", "haddock", "anchovy", "sardine", "fish"],
        "sesame": ["sesame", "tahini", "sesame oil", "sesame seeds"],
        "corn": ["corn", "cornstarch", "maize", "corn syrup", "cornmeal"],
        "mustard": ["mustard", "mustard seeds", "mustard powder"],
        "sulfites": ["sulfite", "sulfites", "wine", "dried fruit", "vinegar"],
        "vegan": ["meat", "chicken", "beef", "pork", "fish", "egg", "milk", "cheese", "butter", "honey"],
        "vegetarian": ["chicken", "beef", "pork", "fish", "seafood", "gelatin"]
    }

    for term in diet_terms:
        expanded_terms.extend(allergy_map.get(term, [term]))

expanded_terms = [t.lower() for t in expanded_terms]

# Filters out any recipe containing an ingredient which falls under the prohibited dietary constraint given.
filtered_indices = []
for idx, text in enumerate(texts):
    if not any(term in text.lower() for term in expanded_terms):
        filtered_indices.append(idx)

if not filtered_indices:
    filtered_indices = list(range(len(texts)))

# Takes remaining recipes and assigns them embeddings from cached embeddings, as well as embedding the query into the same embedding space.
filtered_texts = [texts[i] for i in filtered_indices]
filtered_embeddings = paragraph_embeddings[filtered_indices]

query_embedding = model.encode(query, convert_to_tensor=True)

similarity = util.cos_sim(query_embedding, filtered_embeddings)[0]

# Calculates the cosine similarity between the query embedding and all the filtered recipes embeddings then retrieves the top k (20) most semantically similar recipes.
top_k = 20
top_scores, top_idx_local = torch.topk(similarity, k=min(top_k, len(filtered_texts)))

top_indices = [filtered_indices[i] for i in top_idx_local.tolist()]

# Adjusts the ranking according to nutritional goals, using the recipe metadata.
def multi_criteria_score(row, query, sim_score):
    score = sim_score
    q = query.lower()

    # Tasty boosts recipes that were high in rating.
    if any(w in q for w in ["tasteful", "delicious", "flavourful"]):
        score += row.get("rating", 0) * 0.4

    # Muscle building boosts recipes high in protein and calories.
    if any(w in q for w in ["protein", "muscle", "bulking"]):
        score += row.get("protein", 0) * 0.15

    # Diet/low calorie penalises recipes that are high in calorie.
    if any(w in q for w in ["diet", "low calorie", "healthy"]):
        score -= row.get("calories", 0) * 0.1

    return score

top_candidates = [texts[i] for i in top_indices]

# Applies MCS scores to the 20 candidate recipes.
mcs_scores = []
for local_i, idx in enumerate(top_indices):
    row = recipes.iloc[idx]
    sim_score = similarity[local_i].item()
    mcs_scores.append(multi_criteria_score(row, query, sim_score))

# Re-ranks 20 candidate recipes using a cross-encoder model, and brings back the MCS scoring to preserve nutritional goal fidelity.
cross_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [(query, t) for t in top_candidates]
cross_scores = cross_model.predict(pairs)

combined = []
for i, idx in enumerate(top_indices):
    combined_score = 0.4 * mcs_scores[i] + 0.6 * cross_scores[i]
    combined.append((idx, top_candidates[i], combined_score))

final_ranked = sorted(combined, key=lambda x: x[2], reverse=True)

# Extracts ingredient details from the candidate recipes, with title ingredients given higher weighting.
def extract_all_ingredients(text):
    text_lower = text.lower()

    title = text_lower.split("|")[0]
    title_tokens = set(title.replace(",", " ").split())

    if "features:" in text_lower:
        feature_tokens = set(
            i.strip().lower()
            for i in text_lower.split("features:")[1].split(",")
            if i.strip()
        )
    else:
        feature_tokens = set()

    return title_tokens | feature_tokens

# Calculates ingredient overlap between query and candidate recipes.
def ingredient_match_score(text, query):
    ingredients = extract_all_ingredients(text)

    query_tokens = set(query.lower().split())

    query_ingredients = {
        w for w in query_tokens if w in normalised_ingredients
    }

    return len(query_ingredients.intersection(ingredients))

# Takes top 3 from cross-encoder output.
top_3 = final_ranked[:3]

# Re-ranks candidate recipes according to ingredient matching.
rescored = []

for idx, text, score in top_3:
    match_score = ingredient_match_score(text, query)
    rescored.append((idx, text, score, match_score))

# Sorts primarily by ingredient overlap, then by cross-encoder relecance score if the same number of ingredients matched.
rescored.sort(key=lambda x: (x[3], x[2]), reverse=True)

# Gets final recommended recipe.
best = rescored[0]

# Prints output for checking (this would not be shown in the user's version).
# print("\nFINAL SELECTED RECIPE:\n")
# print(f"Score: {best[2]:.4f}")
# print(f"Ingredient overlap: {best[3]}")
# print(best[1][:500])

# print("\nTOP 5 CROSS-ENCODER RESULTS:\n")
# for rank, (idx, text, score) in enumerate(final_ranked[:5], 1):
#     print(f"Rank {rank} | Score: {score:.4f}")
#     print(text[:200], "\n")


Please tell me any ingredients you would like to use, the type of dish you would like and any nutritional goals you want to adhere to:
eggs, carbonara, diet
Please state all dietary preferences you have (e.g. peanut, gluten-free, vegan). If none, leave blank or say no:
milk, diary


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Recipe generation

In [ ]:
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer, util
import torch
from transformers import pipeline
import os


# Initialize embedding model.
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

# Extract information from the final recommended recipe.
top_idx, top_recipe_text, top_score = final_ranked[0]

# Extract final recommended recipe title.
# Prints output for checking (this would not be shown in the user's version).
top_recipe_name = top_recipe_text.split("|")[0].strip()
# print(f"Top Recipe Name: {top_recipe_name}")


# Extract ingredients from final recommended recipe.
match = re.search(r"Features:\s*(.*)", top_recipe_text)
if match:
    features_str = match.group(1)
    dynamic_ingredients = [ing.strip() for ing in features_str.split(",")]
else:
    print("No 'Features:' found — fallback to empty list.")
    dynamic_ingredients = []

# Prints output for checking (this would not be shown in the user's version).
# print(f"Extracted ingredients: {dynamic_ingredients}")

# Creates/loads embeddings for RecipeNLG dataset (Recipe dataset with instruction.csv).
parsed_path = "/content/drive/MyDrive/Embeddings/parsed_recipe_dataset.pkl"

if os.path.exists(parsed_path):
    new_recipes = pd.read_pickle(parsed_path)
else:
    new_recipes = pd.read_csv("/content/drive/MyDrive/RecipeNLG.csv")
    new_recipes.columns = new_recipes.columns.str.strip()

    # Column names are cleaned and made into strings then made into easily saveable format.
    for col in ['ingredients', 'directions', 'NER']:
        new_recipes[col] = new_recipes[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

    new_recipes.to_pickle(parsed_path)

# Creates/loads embeddings for recipe title embeddings (from dataset Recipe dataset with instruction.csv).
title_embedding_path = "/content/drive/MyDrive/Embeddings/title_embeddings.pt"

# Embeds the final recommended recipe title for cosine similarity search.
top_name_embedding = embedding_model.encode(top_recipe_name, convert_to_tensor=True)
instruction_titles = new_recipes['title'].tolist()

if os.path.exists(title_embedding_path):
    title_embeddings = torch.load(title_embedding_path)
else:
    title_embeddings = embedding_model.encode(
        instruction_titles,
        convert_to_tensor=True
    )
    torch.save(title_embeddings, title_embedding_path)

title_scores = util.cos_sim(top_name_embedding, title_embeddings)[0]

# Finds the top three most similar recipe titles from RecipeNLG to the final recommended recipe title.
n_examples = 3
# Gets the full recipe strings of the top three most similar.
example_indices = torch.topk(title_scores, k=min(n_examples, len(title_scores))).indices.tolist()
example_recipes = new_recipes.iloc[example_indices]

# Prints output for checking (this would not be shown in the user's version).
# print("Selected example recipes for instruction generation:")
# for idx, row in example_recipes.iterrows():
#   print(f" - {row['title']} (Score: {title_scores[idx]:.4f})")

# Constructs few-shot prompting using 3 retrieved example recipes.
examples_text = ""
for i, row in example_recipes.iterrows():
    ingredients = ', '.join(row['ingredients'])
    directions = ' '.join(row['directions'])
    examples_text += f"Example {i+1}:\nIngredients: {ingredients}\nDirections: {directions}\n\n"

# Builds the final prompt for our LLM, with strict prompt engineering.
prompt = f"""
Here are {n_examples} example recipes with ingredients and step-by-step directions:

{examples_text}

The recipe can not include: {diet_input} (if applicable).

Now generate step-by-step instructions for a new recipe with ingredients: {', '.join(dynamic_ingredients)},
with the same title as {top_recipe_name}.

First give the title, then list ingredients, then list the directions.

Give directions in steps.

Do not reference the recipe number.

Do not comment on the style / tone that you use when generating the recipe.

"""

# Generates the final recipe from the retrieval-augmented prompt.
generator = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")

output = generator(
    prompt,
    max_new_tokens=450,
    temperature=0.7,
    return_full_text=False
)

print("Your recommended recipe:\n")

generated_recipe = output[0]['generated_text']

print(generated_recipe)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Your recommended recipe:

Here is the recipe:

Fresh Egg Semolina Fettucine

Ingredients:

* 2 large eggs
* 1 cup semolina
* 2 cups water
* 1 tsp salt

Directions:

Step 1: In a medium saucepan, bring the water to a boil over high heat.
Step 2: Gradually add the semolina, stirring constantly to prevent lumps.
Step 3: Reduce heat to medium-low and cook for 5 minutes, stirring frequently, until the mixture is smooth and free of lumps.
Step 4: Remove the saucepan from the heat and stir in the salt.
Step 5: Allow the mixture to cool slightly, then stir in the beaten eggs.
Step 6: Gradually add the eggs to the saucepan, stirring constantly to prevent the eggs from scrambling.
Step 7: Cook for an additional 2-3 minutes, until the eggs are just set.
Step 8: Remove the saucepan from the heat and let it cool slightly.
Step 9: Serve the fettucine hot, topped with additional egg if desired.


Evaluation, both human and LLM-as-a-judge

In [ ]:
import json
import os
import numpy as np
import re
from transformers import pipeline

HUMAN_EVAL_PATH = "/content/drive/MyDrive/human_evaluated_recipes.json"
LLM_EVAL_PATH = "/content/drive/MyDrive/llm_evaluated_recipes.json"

# Creates functions for loading and saving both human and LLM evaluations.
def load_human_evals():
    if os.path.exists(HUMAN_EVAL_PATH):
        with open(HUMAN_EVAL_PATH, "r") as f:
            return json.load(f)
    return []

def save_human_evals(evals):
    with open(HUMAN_EVAL_PATH, "w") as f:
        json.dump(evals, f, indent=2)

def load_llm_evals():
    if os.path.exists(LLM_EVAL_PATH):
        with open(LLM_EVAL_PATH, "r") as f:
            return json.load(f)
    return []

def save_llm_evals(evals):
    with open(LLM_EVAL_PATH, "w") as f:
        json.dump(evals, f, indent=2)

# Creates the recipe ID, checking for the highest and adding one.
def get_next_recipe_id():
    evals = load_human_evals()
    if not evals:
        return 0
    return max(e["recipe_id"] for e in evals) + 1

# Asks the human (user) to score the recipe based on the 5 criteria.
def human_evaluate_and_edit(recipe_text, ingredients, recipe_id):
    print("\n" + "="*50)
    print("RECIPE TO EVALUATE:\n")
    print(recipe_text)
    print("="*50 + "\n")

    # --- Step 1: Human scores ---
    while True:
        try:
            fluency = int(input("Fluency (1-5): "))
            relevance = int(input("Relevance (1-5): "))
            correctness = int(input("Correctness / Safety (1-5): "))
            structure = int(input("Structure (1-5): "))
            taste = int(input("Taste plausibility (1-5): "))

            if all(1 <= x <= 5 for x in [fluency, relevance, correctness, structure, taste]):
                break
        except:
            pass
        print("Enter integers 1–5")

    human_scores = {
        "fluency": fluency,
        "relevance": relevance,
        "correctness": correctness,
        "structure": structure,
        "taste": taste,
        "human_score": np.mean([fluency, relevance, correctness, structure, taste])
    }
    print("\nScores saved:", human_scores)

    # Invites the human (user) to edit the recipe to improve it.
    print("\nEdit the recipe below (press Enter on an empty line to finish):")
    print("Original Recipe:\n")
    print(recipe_text)
    print("End of Original Recipe.\n")

    improved_lines = []
    while True:
        line = input()
        if line.strip() == "":
            break
        improved_lines.append(line)

    improved_recipe = "\n".join([l for l in improved_lines if l.strip() != ""]) or recipe_text

    # Saves the human evaluation.
    human_entry = {
        "recipe_id": recipe_id,
        "ingredients": ingredients,
        "original": recipe_text,
        "improved": improved_recipe,
        "scores": human_scores
    }

    evals = load_human_evals()
    evals.append(human_entry)
    save_human_evals(evals)

    print(f"\nHuman evaluation saved with recipe_id {recipe_id}")
    return human_entry

# Creates an LLM-as-a-judge to evaluate the recipe.
judge = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B-Instruct"
)

# Uses strict prompt engineering to get the LLM to evaluate the recipe according to the 5 criteria.
def build_judge_prompt(recipe_text, ingredients, query):
    return f"""
You are a strict recipe evaluator. You do NOT generate recipes.

Evaluate the recipe using these criteria (score 1–5) in this format:
 {{"fluency": X, "relevance": X, "correctness": X, "structure": X, "taste": X}}:
- fluency: clarity and readability
- relevance: matches query and ingredients
- correctness: safe and logically correct cooking steps
- structure: clear ingredients + step-by-step format
- taste: likely to taste good

IMPORTANT:
- Return EXACTLY ONE JSON object
- Do NOT explain
- Do NOT repeat
- Do NOT output multiple answers

TEXT TO EVALUATE:
---------------------
{recipe_text}
---------------------

Ingredients:
{', '.join(ingredients)}

Query:
{query}
"""

# Finds the JSON the LLM-as-a-judge gave.
def extract_first_json(text):
    matches = re.findall(r"\{[\s\S]*?\}", text)
    for match in matches:
        try:
            return json.loads(match)
        except:
            continue
    return None

# Runs the LLM-as-a-judge and gets the resultant score.
def single_llm_judge(recipe_text, ingredients, query):
    prompt = build_judge_prompt(recipe_text, ingredients, query)
    output = judge(
        prompt,
        max_new_tokens=100,
        temperature=0.0,
        do_sample=False,
        return_full_text=False
    )
    raw_text = output[0]["generated_text"]
    scores = extract_first_json(raw_text)
    if scores is None:
        print("Failed to parse LLM output:")
        print(raw_text)
        return None
    return scores

# This allows for future reruns of the LLM-as-a-judge to get a more accurate score.
def llm_judge_recipe(recipe_text, ingredients, query, n_runs=3):
    all_scores = []
    for _ in range(n_runs):
        scores = single_llm_judge(recipe_text, ingredients, query)
        if scores:
            all_scores.append(scores)
    if not all_scores:
        print("All LLM judge runs failed.")
        return None
    keys = ["fluency", "relevance", "correctness", "structure", "taste"]
    avg_scores = {k: float(np.mean([s[k] for s in all_scores if k in s])) for k in keys}
    avg_scores["llm_score"] = float(np.mean(list(avg_scores.values())))
    avg_scores["runs"] = len(all_scores)
    print("\nLLM Judge (Averaged):")
    print(json.dumps(avg_scores, indent=2))
    return avg_scores

# Defines a function to run the LLM-as-a-judge and saves its output, while throwing errors if there is already a recipe ID saved or if it didn't work.
def llm_judge_and_save(recipe_text, ingredients, query, recipe_id, n_runs=3):
    scores = llm_judge_recipe(recipe_text, ingredients, query, n_runs=n_runs)
    if scores is None:
        print("LLM evaluation failed — not saving.")
        return None
    evals = load_llm_evals()
    if any(e["recipe_id"] == recipe_id for e in evals):
        print(f"LLM eval already exists for recipe_id {recipe_id}, skipping save.")
        return None
    llm_entry = {
        "recipe_id": recipe_id,
        "ingredients": ingredients,
        "original": recipe_text,
        "llm_scores": scores
    }
    evals.append(llm_entry)
    save_llm_evals(evals)
    print(f"LLM evaluation saved with recipe_id {recipe_id}")
    return llm_entry

# Generates the new recipe ID.
recipe_id = get_next_recipe_id()

# Runs the LLM-as-a-judge then stores its score.
llm_entry = llm_judge_and_save(
    generated_recipe,
    dynamic_ingredients,
    query,
    recipe_id,
    n_runs=1
)

# Runs the pipeline for human evaluation (for the same recipe as the LLM) and saved the output.
human_entry = human_evaluate_and_edit(
    generated_recipe,
    dynamic_ingredients,
    recipe_id
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



LLM Judge (Averaged):
{
  "fluency": 5.0,
  "relevance": 4.0,
  "correctness": 5.0,
  "structure": 4.0,
  "taste": 3.0,
  "llm_score": 4.2,
  "runs": 1
}
LLM eval already exists for recipe_id 118, skipping save.

RECIPE TO EVALUATE:

Here is the recipe:

Fresh Egg Semolina Fettucine

Ingredients:

* 2 large eggs
* 1 cup semolina
* 2 cups water
* 1 tsp salt

Directions:

Step 1: In a medium saucepan, bring the water to a boil over high heat.
Step 2: Gradually add the semolina, stirring constantly to prevent lumps.
Step 3: Reduce heat to medium-low and cook for 5 minutes, stirring frequently, until the mixture is smooth and free of lumps.
Step 4: Remove the saucepan from the heat and stir in the salt.
Step 5: Allow the mixture to cool slightly, then stir in the beaten eggs.
Step 6: Gradually add the eggs to the saucepan, stirring constantly to prevent the eggs from scrambling.
Step 7: Cook for an additional 2-3 minutes, until the eggs are just set.
Step 8: Remove the saucepan from the h